In [19]:
import math

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Basic Pytorch syntax

In [ ]:
torch.rand(5,5) # Creation of random tensors

tensor([[0.8569, 0.1751, 0.7588, 0.9353, 0.3978],
        [0.1434, 0.0880, 0.0038, 0.8500, 0.0171],
        [0.0937, 0.0399, 0.1069, 0.0472, 0.3945],
        [0.8294, 0.9741, 0.3101, 0.1874, 0.1502],
        [0.9077, 0.7337, 0.7694, 0.0943, 0.1370]])

In [33]:
a=torch.rand(5,5,5)
a.shape

torch.Size([5, 5, 5])

In [ ]:
a=[1,2,3,4,5]
a=torch.tensor(a)
a # Conversion of lists to tensors

tensor([1, 2, 3, 4, 5])

In [38]:
X=nn.Linear(5,5) # Initializes a matrix of shape (5,5) with bias term, 
X=nn.Linear(5,5, bias=False)  # samething but without the bias term, essentially just the matrix

In [40]:
a=torch.rand(10,5)
X(a).shape  # Gives the output a.X, thus the output is of size (10,5)

torch.Size([10, 5])

# Transformers components
* Tokenization (Python Dict)
    * Converts every word in the input to a specific integer, usually a direct map
    * Types are subword, word, multi-word level
* Vectorization (class torch.nn.Emebdding, converts a word to its corresponding vector)
    * Each vector for a word in a high dimensional vector space is used to represent that word
    * Similar words lie closer to each other 
    * It encodes the semantic meaning of that word      
* Attention layer (Multi-head attention)
    * Sentences as a whole have a greater meaning than its component words, this is computed by the attention mechanism 
    * The importance of every word w.r.t every other word is calculated in this to get a more effective understanding of the input sentence
* Multi-Layer-Perceptron / Feed-Forward-Layer
    * A basic neural network that expands the dimensionality of the input and reduces it again
    * Acts like the memory of the transformer, imagine how LLMs memorize that 1+1=2
    * The larger dimensionality means that more information can be stored and be attended by smaller sequences
* Residual Connections
    * The attention layers outputs and the 

# Attention Mechanism

The scaled dot-product attention is defined as:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

Where:
- \(Q\): Query matrix.
- \(K\): Key matrix.
- \(V\): Value matrix.
- \(d_k\): Dimensionality of the keys.

---

## Matrix Sizes

### Input Dimensions
- Sequence length: \(T\)
- Embedding size: \(d_{\text{model}}\)
- Batch size: \(B\)

### Derived Matrices
- Queries:
  $$
  Q = XW_Q \in \mathbb{R}^{B \times T \times d_k}
  $$
- Keys:
  $$
  K = XW_K \in \mathbb{R}^{B \times T \times d_k}
  $$
- Values:
  $$
  V = XW_V \in \mathbb{R}^{B \times T \times d_v}
  $$

### Attention Score Matrix
- Dot-product of \(Q\) and \(K^\top\):  
  $$
  QK^\top \in \mathbb{R}^{B \times T \times T}
  $$  
  Each element represents the similarity between queries and keys.

### Softmax
- Softmax is applied row-wise along the last dimension:  
  $$
  \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) \in \mathbb{R}^{B \times T \times T}
  $$

### Weighted Values
- The attention weights are multiplied with \(V\):  

  $$
  \text{Attention}(Q, K, V) \in \mathbb{R}^{B \times T \times d_v}
  $$

---

## Multi-Head Attention

Multi-head attention is defined as:

$$
\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W_O
$$

Where each head computes:

$$
\text{head}_i = \text{Attention}(QW_Q^i, KW_K^i, VW_V^i)
$$
### Dimensionality of Heads

- Number of heads: \( h \)
- Dimensionality per head:
  $$
  d_k = \frac{d_{\text{model}}}{h}, \quad d_v = \frac{d_{\text{model}}}{h}
  $$
- Sizes of learned parameters:
  $$
  W_Q^i, W_K^i, W_V^i \in \mathbb{R}^{d_{\text{model}} \times d_k}
  $$

  $$
  W_O \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}
  $$


## What each matrice signifies in the attention mechanism
* Query   : Analogous to the query in a google search
* Key     : Analogous to the titles of webpages       
* Value   : Analogous to the content of thoese webpages 

## Important concepts within transformers
* Vector space: size of the vector that represents each and every word
* Attention(heads) : creates a matrix of size (sequence, sequence) regarding similarity between each word in the sequence for a specific topic
* Multi-head attention : creates multiple attention matrices for different topics

In [12]:
x = torch.rand(32, 128, 256) # batch, sequence, embedding

In [13]:
x.shape

torch.Size([32, 128, 256])

In [4]:
d_model=256
d_k=64
d_v=64


In [5]:
Wq=nn.Linear(d_model, d_k, bias=False)
Wk=nn.Linear(d_model, d_k, bias=False)
Wv=nn.Linear(d_model, d_v, bias=False)

In [14]:
Q=Wk(x)
K=Wq(x)
V=Wv(x)

In [16]:
K.shape

torch.Size([32, 128, 64])

In [20]:
gating_mat=(Q@K.transpose(-1, -2))/math.sqrt(d_k)

In [21]:
gating_mat.shape

torch.Size([32, 128, 128])

In [ ]:
gating_mat=F.softmax(gating_mat, dim=-1) # converts the inputs to a probability distribution

In [ ]:
attention=(gating_mat@V)

In [30]:
FFN=nn.Sequential(
    nn.Linear(d_model, d_model*4),
    nn.ReLU(),
    nn.Linear(d_model*4, d_model)
)

In [ ]:
FFN(attention)

In [57]:
class Head(nn.Module):
    def __init__(self, d_model, d_k, d_v):
        super(Head, self).__init__()
        # Attention section
        self.Wq=nn.Linear(d_model, d_k, bias=False) # Computes the query matrix for the given input 
        self.Wk=nn.Linear(d_model, d_k, bias=False) # Computes the key matrix for the given input
        self.Wv=nn.Linear(d_model, d_v, bias=False) # Computes the value matrix for the given input
        self.Wo=nn.Linear(d_v, d_model, bias=False) # Converts the output back to the desired shape
        self.d_k=d_k
        self.d_v=d_v

        # Feed forward section
        self.FFN=nn.Sequential(
            nn.Linear(d_model, d_model*4),
            nn.ReLU(),
            nn.Linear(d_model*4, d_model)
        )

    def forward(self, x):
        Q=self.Wq(x)
        K=self.Wk(x)
        V=self.Wv(x)
        gating_mat=(Q@K.transpose(-1, -2))/math.sqrt(d_k)
        gating_mat=F.softmax(gating_mat, dim=-1)
        attention= F.normalize(self.Wo(gating_mat@V) +x, dim=-1)

        logits = F.normalize(self.FFN(attention)+attention)
        return logits

In [58]:
x=torch.rand(5,16,128)
layer=Head(128,64,64)

In [59]:
layer(x).shape

torch.Size([5, 16, 128])

In [76]:
class MultiHead(nn.Module):
    def __init__(self, d_model, d_v, d_k, n_heads):
        super(MultiHead, self).__init__()
        self.d_v=d_v
        self.d_k=d_v
        self.n_heads=n_heads
        # Self Attention layer
        self.Wq=nn.ModuleList([nn.Linear(d_model, (int)(d_k/n_heads)) for _ in range(n_heads)])
        self.Wk=nn.ModuleList([nn.Linear(d_model, (int)(d_k/n_heads)) for _ in range(n_heads)])
        self.Wv=nn.ModuleList([nn.Linear(d_model, (int)(d_v/n_heads)) for _ in range(n_heads)])
        self.Wo=nn.Linear(d_v, d_model)

        # MLP
        self.FFN=nn.Sequential(
            nn.Linear(d_model, d_model*4),
            nn.ReLU(),
            nn.Linear(d_model*4, d_model)
        )

    def forward(self,x):
        Q=[Wq(x) for Wq in self.Wq]
        K=[Wk(x) for Wk in self.Wk]
        V=[Wv(x) for Wv in self.Wv]
        gating_matrix=[Q[i]@K[i].transpose(-1,-2)/math.sqrt(d_k) for i in range(self.n_heads)]
        gating_matrix=[F.softmax(mat, dim=-1) for mat in gating_matrix]
        attentions=[gating_matrix[i]@V[i] for i in range(self.n_heads)]
        attention=torch.concat(attentions, dim=-1)
        
        attention=F.normalize(self.Wo(attention)+x)
        logits = F.normalize(self.FFN(attention)+attention)
        return logits

In [77]:
x=torch.rand(5,16,128)
layer=MultiHead(128,64,64, 4)

In [79]:
layer(x).shape

torch.Size([5, 16, 128])